In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R

m = 1.6                     # mass (kg)
g = 9.81                    # gravity
Ixx, Iyy, Izz = 0.025, 0.025, 0.045
I = np.diag([Ixx, Iyy, Izz])
I_inv = np.linalg.inv(I)

N = 1197
T = 15
time = np.linspace(0, T, N)
dt = time[1] - time[0]

pos = np.zeros((N,3))
vel = np.zeros((N,3))
omega = np.zeros((N,3))
ang_acc = np.zeros((N,3))
quat = np.zeros((N,4))   # (w,x,y,z)
acc_body = np.zeros((N,3))
u = np.zeros((N,4))

quat[0] = [1,0,0,0]   # initial orientation

for i, t in enumerate(time):

    thrust = m*g \
             + 2.0*np.sin(0.6*t) \
             + 1.5*np.sin(1.3*t)

    tau_x = 0.04*np.sin(0.9*t) + 0.02*np.sin(2.2*t)
    tau_y = 0.035*np.cos(0.7*t) + 0.015*np.sin(1.8*t)
    tau_z = 0.03*np.sin(0.5*t) + 0.01*np.cos(2.0*t)

    u[i] = [thrust, tau_x, tau_y, tau_z]

for i in range(N-1):

    rot = R.from_quat([quat[i,1], quat[i,2], quat[i,3], quat[i,0]])
    R_mat = rot.as_matrix()

    thrust_body = np.array([0, 0, u[i,0]])
    force_world = R_mat @ thrust_body - np.array([0,0,m*g])
    acc_world = force_world / m

    vel[i+1] = vel[i] + acc_world * dt
    pos[i+1] = pos[i] + vel[i] * dt

    tau = u[i,1:]

    ang_acc[i] = I_inv @ (tau - np.cross(omega[i], I @ omega[i]))
    omega[i+1] = omega[i] + ang_acc[i] * dt

    wx, wy, wz = omega[i]

    Omega = np.array([
        [0, -wx, -wy, -wz],
        [wx, 0, wz, -wy],
        [wy, -wz, 0, wx],
        [wz, wy, -wx, 0]
    ])

    quat_dot = 0.5 * Omega @ quat[i]
    quat[i+1] = quat[i] + quat_dot * dt
    quat[i+1] /= np.linalg.norm(quat[i+1])  # normalize

for i in range(N):
    rot = R.from_quat([quat[i,1], quat[i,2], quat[i,3], quat[i,0]])
    R_mat = rot.as_matrix()

    if i > 0:
        acc_world = (vel[i] - vel[i-1]) / dt
    else:
        acc_world = np.zeros(3)

    acc_body[i] = R_mat.T @ acc_world

aux = np.column_stack([
    np.sin(0.2*time),
    np.sin(0.4*time),
    np.sin(0.6*time),
    np.cos(0.3*time),
    np.cos(0.5*time),
    np.sin(0.8*time)
])

landed = np.zeros(N)
landed[-80:] = 1

df = pd.DataFrame({
    "sno": np.arange(N),
    "timestamp": time,
    "u0": u[:,0],
    "u1": u[:,1],
    "u2": u[:,2],
    "u3": u[:,3],
    "vx": vel[:,0],
    "vy": vel[:,1],
    "vz": vel[:,2],
    "q0": quat[:,0],
    "q1": quat[:,1],
    "q2": quat[:,2],
    "q3": quat[:,3],
    "ang_vel_x": omega[:,0],
    "ang_vel_y": omega[:,1],
    "ang_vel_z": omega[:,2],
    "acc_b_x": acc_body[:,0],
    "acc_b_y": acc_body[:,1],
    "acc_b_z": acc_body[:,2],
    "ang_acc_b_x": ang_acc[:,0],
    "ang_acc_b_y": ang_acc[:,1],
    "ang_acc_b_z": ang_acc[:,2],
    "old_index": np.arange(N),
    "aux1": aux[:,0],
    "aux2": aux[:,1],
    "aux3": aux[:,2],
    "aux4": aux[:,3],
    "aux5": aux[:,4],
    "aux6": aux[:,5],
    "landed": landed
})

df.to_excel("quadrotor_model_trajectory_synthetic_dense.xlsx", index=False)

print("Synthetic dense physics-consistent dataset generated.")
print("Total rows:", len(df))

Synthetic dense physics-consistent dataset generated.
Total rows: 1197
